# Notebook 2: Create the Labels

This notebook builds the late-delivery label by comparing the actual delivery date to the estimated delivery date, for delivered orders only. It checks the label on a few real orders and looks at the class distribution to detect imbalance.

**Reads:** `ml_table.csv`  
**Artifact produced:** `labeled_table.csv`

In [1]:
import pandas as pd

In [2]:
ml_table = pd.read_csv("ml_table.csv", parse_dates=[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
])

In [3]:
print("Loaded ml_table:", ml_table.shape)

Loaded ml_table: (99441, 20)


In [4]:
# Only orders that were actually delivered have a real delivery date to compare.
print("\nOrder status counts:")
print(ml_table["order_status"].value_counts())


Order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [5]:
delivered = ml_table[ml_table["order_status"] == "delivered"].copy()
print("\nDelivered orders:", delivered.shape)
print("Missing delivered date among delivered orders:", delivered["order_delivered_customer_date"].isna().sum())


Delivered orders: (96478, 20)
Missing delivered date among delivered orders: 8


In [6]:
delivered = delivered.dropna(subset=["order_delivered_customer_date", "order_estimated_delivery_date"])

In [7]:
# Build the label: 1 = late, 0 = on time
delivered["is_late"] = (delivered["order_delivered_customer_date"] > delivered["order_estimated_delivery_date"]).astype(int)

In [8]:
# Sanity check the label on a few real orders
print("\nSample check:")
print(delivered[["order_id", "order_delivered_customer_date", "order_estimated_delivery_date", "is_late"]].head(10))


Sample check:
                            order_id order_delivered_customer_date  \
0   e481f51cbdc54678b7cc49136f2d6af7           2017-10-10 21:25:13   
1   53cdb2fc8bc7dce0b6741e2150273451           2018-08-07 15:27:45   
2   47770eb9100c2d0c44946d9cf07ec65d           2018-08-17 18:06:29   
3   949d5b44dbf5de918fe9c16f97b45f8a           2017-12-02 00:28:42   
4   ad21c59c0840e6cb83a9ceb5573f8159           2018-02-16 18:17:02   
5   a4591c265e18cb1dcee52889e2d8acc3           2017-07-26 10:57:55   
7   6514b8ad8028c9f2cc2374ded245783f           2017-05-26 12:55:51   
8   76c6e866289321a7c93b82b54852dc33           2017-02-02 14:08:10   
9   e69bfb5eb88e0ed6a785585b27e16dbf           2017-08-16 17:14:30   
10  e6ce16cb79ec1d90b1da9085a6118aeb           2017-05-29 11:18:31   

   order_estimated_delivery_date  is_late  
0                     2017-10-18        0  
1                     2018-08-13        0  
2                     2018-09-04        0  
3                     2017-12-15      

In [9]:
# Class distribution
print("\nClass distribution:")
print(delivered["is_late"].value_counts())
print(delivered["is_late"].value_counts(normalize=True))


Class distribution:
is_late
0    88644
1     7826
Name: count, dtype: int64
is_late
0    0.918876
1    0.081124
Name: proportion, dtype: float64


In [10]:
late_ratio = delivered["is_late"].mean()
print(f"\nLate ratio: {late_ratio:.3f}")
if late_ratio < 0.2 or late_ratio > 0.8:
    print("This looks like an imbalanced classification problem.")
else:
    print("Classes are reasonably balanced.")


Late ratio: 0.081
This looks like an imbalanced classification problem.


In [11]:
# Save artifact: the labeled table
delivered.to_csv("labeled_table.csv", index=False)
print("\nArtifact saved: labeled_table.csv")
print("Final labeled table shape:", delivered.shape)


Artifact saved: labeled_table.csv
Final labeled table shape: (96470, 21)
